# Notebook 3: Evaluation, Explainability & Demo
## AI-Generated Face Detection

**Purpose:**
1. Full test set evaluation (known + unseen generators)
2. Cross-generator generalisation analysis (Flux holdout)
3. JPEG compression robustness testing
4. GradCAM explainability visualisations
5. Gradio web demo interface

**Prerequisites:** Notebooks 1 and 2 completed. Checkpoints at:  
`/content/drive/MyDrive/deepfake_detection/checkpoints/`

## 1. Setup & Load Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install timm albumentations scipy scikit-learn grad-cam gradio

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import io
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from torchvision import transforms
import timm
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_curve
)
from scipy.fft import dctn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Paths
PROJECT_DIR = '/content/drive/MyDrive/deepfake_detection'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_dataset')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# =============================================
# Recreate model architectures (same as Notebook 2)
# =============================================

class ViTDeepfakeDetector(nn.Module):
    def __init__(self, freeze_blocks=18):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_large_patch14_clip_224', pretrained=False, num_classes=0
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def extract_features(self, x):
        features = self.backbone(x)
        x = self.head[0](features)
        x = self.head[1](x)
        return x


class FrequencyBranchDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, num_classes=0
        )
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(
            1, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None
        )
        self.backbone.conv_stem = new_conv
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def extract_features(self, x):
        features = self.backbone(x)
        x = self.head[0](features)
        x = self.head[1](x)
        return x


class MetaClassifier(nn.Module):
    def __init__(self, input_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, x):
        return self.net(x)

    def predict_calibrated(self, x):
        logits = self.net(x)
        return torch.sigmoid(logits / self.temperature)

print('Model classes defined.')

In [ ]:
# Load trained checkpoints
print('Loading trained models...')

# Using weights_only=False to support legacy checkpoints containing numpy scalars
vit_model = ViTDeepfakeDetector().to(device)
vit_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'vit_best.pth'), map_location=device, weights_only=False)
vit_model.load_state_dict(vit_ckpt['model_state_dict'])
vit_model.eval()
print(f'  ViT loaded — trained epoch {vit_ckpt["epoch"]}, val AUC: {vit_ckpt["val_auc"]:.4f}')

freq_model = FrequencyBranchDetector().to(device)
freq_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'freq_best.pth'), map_location=device, weights_only=False)
freq_model.load_state_dict(freq_ckpt['model_state_dict'])
freq_model.eval()
print(f'  Freq loaded — trained epoch {freq_ckpt["epoch"]}, val AUC: {freq_ckpt["val_auc"]:.4f}')

meta_model = MetaClassifier().to(device)
meta_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'meta_calibrated.pth'), map_location=device, weights_only=False))
meta_model.eval()
print(f'  Meta-classifier loaded (temperature: {meta_model.temperature.item():.4f})')

print('\nAll models loaded and set to eval mode.')

In [ ]:
# Dataset and DataLoader for test set
import albumentations as A
from albumentations.pytorch import ToTensorV2

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    def __init__(self, manifest_df, data_root, split='test', augment=None):
        self.data_root = data_root
        self.df = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rgb = np.array(Image.open(os.path.join(self.data_root, row['crop_path'])).convert('RGB'))
        if self.augment:
            rgb = self.augment(image=rgb)['image']
        else:
            rgb = val_transform(image=rgb)['image']
        dct = np.load(os.path.join(self.data_root, row['dct_path']))
        dct = torch.tensor(dct, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return rgb, dct, label

manifest_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'manifest.csv'))

test_ds = DeepfakeDataset(manifest_df, PROCESSED_DIR, split='test')
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Test set size: {len(test_ds)}')
print(f'Test set composition:')
test_df = manifest_df[manifest_df['split'] == 'test']
print(test_df.groupby(['category', 'generator']).size().to_string())

---
## 2. Full Test Set Evaluation

Evaluate all three models (ViT, Frequency, Fused) on the complete test set.

In [ ]:
@torch.no_grad()
def full_evaluation(vit_model, freq_model, meta_model, loader, device):
    """
    Run all three models on the data and return predictions + labels.
    """
    vit_preds, freq_preds, fused_preds = [], [], []
    all_labels = []

    for rgb, dct, labels in tqdm(loader, desc='Evaluating'):
        rgb = rgb.to(device)
        dct = dct.to(device)

        with autocast():
            # ViT predictions
            vit_logits = vit_model(rgb).squeeze(1)
            vit_probs = torch.sigmoid(vit_logits).cpu().numpy()

            # Frequency predictions
            freq_logits = freq_model(dct).squeeze(1)
            freq_probs = torch.sigmoid(freq_logits).cpu().numpy()

            # Fused predictions
            vit_feat = vit_model.extract_features(rgb).float()
            freq_feat = freq_model.extract_features(dct).float()
            fused_feat = torch.cat([vit_feat, freq_feat], dim=1)
            fused_probs = meta_model.predict_calibrated(fused_feat).squeeze(1).cpu().numpy()

        vit_preds.extend(vit_probs)
        freq_preds.extend(freq_probs)
        fused_preds.extend(fused_probs)
        all_labels.extend(labels.numpy())

    return {
        'labels': np.array(all_labels),
        'vit': np.array(vit_preds),
        'freq': np.array(freq_preds),
        'fused': np.array(fused_preds),
    }

# Run evaluation
results = full_evaluation(vit_model, freq_model, meta_model, test_loader, device)
print(f'\nEvaluation complete: {len(results["labels"])} test samples')

In [ ]:
# Compute metrics for all three models

def compute_metrics(labels, preds, threshold=0.5):
    binary_preds = (preds >= threshold).astype(int)
    return {
        'AUC': roc_auc_score(labels, preds),
        'Accuracy': accuracy_score(labels, binary_preds),
        'Precision': precision_score(labels, binary_preds, zero_division=0),
        'Recall': recall_score(labels, binary_preds, zero_division=0),
        'F1': f1_score(labels, binary_preds, zero_division=0),
    }

labels = results['labels']

# Compute metrics once
vit_m   = compute_metrics(labels, results['vit'])
freq_m  = compute_metrics(labels, results['freq'])
fused_m = compute_metrics(labels, results['fused'])

# FIX: consistent column widths
col_w = 16
header = f'{"Metric":<12s}' + f'{"ViT (Branch 1)":>{col_w}s}' + f'{"Freq (Branch 2)":>{col_w}s}' + f'{"Fused":>{col_w}s}'
divider = '=' * len(header)

print(divider)
print('TEST SET RESULTS')
print(divider)
print(header)
print('-' * len(header))
for metric in ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']:
    print(f'{metric:<12s}' + f'{vit_m[metric]:>{col_w}.4f}' + f'{freq_m[metric]:>{col_w}.4f}' + f'{fused_m[metric]:>{col_w}.4f}')
print(divider)


In [ ]:
# ROC Curves
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

for name, preds, color in [
    ('ViT (Branch 1)', results['vit'], '#2196F3'),
    ('Freq (Branch 2)', results['freq'], '#FF9800'),
    ('Fused', results['fused'], '#4CAF50'),
]:
    fpr, tpr, _ = roc_curve(labels, preds)
    auc = roc_auc_score(labels, preds)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Test Set', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=150)
plt.show()

In [ ]:
# Confusion Matrix for fused model
fused_binary = (results['fused'] >= 0.5).astype(int)
cm = confusion_matrix(labels, fused_binary)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

# Add text annotations
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, f'{cm[i,j]}', ha='center', va='center', fontsize=20, color=color)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Real', 'Fake'])
ax.set_yticklabels(['Real', 'Fake'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — Fused Model', fontsize=14)
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

print('\nClassification Report (Fused Model):')
print(classification_report(labels, fused_binary, target_names=['Real', 'Fake']))

---
## 3. Cross-Generator Generalisation

Evaluate performance per generator across all three generator types (all seen during training)..  
This is the most critical evaluation — it tests whether the model detects general forensic signals vs. generator-specific shortcuts.

In [ ]:
# Per-generator evaluation

test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)

# We need per-sample generator info alongside predictions
# Re-run evaluation capturing the order
generators = test_df['generator'].values
categories = test_df['category'].values

print('=' * 70)
print('CROSS-GENERATOR EVALUATION (Fused Model)')
print('=' * 70)
print(f'{"Generator":<20s} {"Count":>6s} {"AUC":>8s} {"Acc":>8s} {"Prec":>8s} {"Recall":>8s} {"Note"}')
print('-' * 75)

cross_gen_results = {}

for gen in ['none', 'stable_diffusion', 'z_image', 'flux']:
    mask = generators == gen
    if mask.sum() == 0:
        continue

    gen_labels = labels[mask]
    gen_preds = results['fused'][mask]

    # For real images (gen='none'), all labels are 0
    # AUC needs both classes, so we skip AUC for single-class subsets
    gen_binary = (gen_preds >= 0.5).astype(int)
    acc = accuracy_score(gen_labels, gen_binary)

    if len(np.unique(gen_labels)) > 1:
        auc = roc_auc_score(gen_labels, gen_preds)
        prec = precision_score(gen_labels, gen_binary, zero_division=0)
        rec = recall_score(gen_labels, gen_binary, zero_division=0)
        auc_str = f'{auc:.4f}'
        prec_str = f'{prec:.4f}'
        rec_str = f'{rec:.4f}'
    else:
        auc_str = 'N/A'
        prec_str = 'N/A'
        rec_str = f'{acc:.4f}'

    note = '(real images)' if gen == 'none' else '(seen in training)'
    display_name = gen if gen != 'none' else 'Real Photos'

    print(f'{display_name:<20s} {mask.sum():>6d} {auc_str:>8s} {acc:>8.4f} {prec_str:>8s} {rec_str:>8s} {note}')

    cross_gen_results[gen] = {
        'count': int(mask.sum()),
        'accuracy': acc,
        'preds': gen_preds.tolist()
    }

print('=' * 75)

In [ ]:
# Prediction distribution per generator
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

gen_names = ['none', 'stable_diffusion', 'z_image', 'flux']
display_names = ['Real Photos', 'Stable Diffusion', 'Z-Image', 'Flux']
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']

for idx, (gen, display, color) in enumerate(zip(gen_names, display_names, colors)):
    mask = generators == gen
    if mask.sum() == 0:
        axes[idx].set_title(f'{display}\n(no samples)')
        continue

    preds = results['fused'][mask]
    axes[idx].hist(preds, bins=20, range=(0, 1), color=color, alpha=0.7, edgecolor='black')
    axes[idx].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
    axes[idx].set_title(f'{display}\n(n={mask.sum()})', fontsize=11)
    axes[idx].set_xlabel('P(Fake)')
    axes[idx].set_ylabel('Count')
    axes[idx].set_xlim(0, 1)

plt.suptitle('Fused Model Prediction Distribution by Generator', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cross_generator_distributions.png'), dpi=150)
plt.show()

---
## 4. JPEG Compression Robustness

Test how the model holds up when images are JPEG-compressed at various quality levels.  
Real-world images are almost always recompressed by social media platforms.

In [ ]:
def apply_jpeg_compression(image_array, quality):
    """Apply JPEG compression to an image array and return the result."""
    img = Image.fromarray(image_array)
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    return np.array(Image.open(buffer).convert('RGB'))


def compute_dct(image_array):
    gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY).astype(np.float64)
    dct_coeffs = dctn(gray, norm='ortho')
    log_dct = np.log(1 + np.abs(dct_coeffs))
    dct_min, dct_max = log_dct.min(), log_dct.max()
    if dct_max - dct_min > 1e-8:
        log_dct = (log_dct - dct_min) / (dct_max - dct_min)
    else:
        log_dct = np.zeros_like(log_dct)
    return log_dct.astype(np.float32)


class CompressedTestDataset(Dataset):
    """Test dataset with JPEG compression applied on-the-fly."""
    def __init__(self, manifest_df, data_root, jpeg_quality=100):
        self.data_root = data_root
        self.df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)
        self.jpeg_quality = jpeg_quality

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rgb = np.array(Image.open(os.path.join(self.data_root, row['crop_path'])).convert('RGB'))

        # Apply JPEG compression
        if self.jpeg_quality < 100:
            rgb = apply_jpeg_compression(rgb, self.jpeg_quality)

        # Recompute DCT from compressed image
        dct = compute_dct(rgb)

        rgb = val_transform(image=rgb)['image']
        dct = torch.tensor(dct, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.float32)

        return rgb, dct, label

print('Compression robustness classes ready.')

In [ ]:
# Run compression robustness test

jpeg_qualities = [100, 90, 80, 70, 60]
compression_results = {}

for quality in jpeg_qualities:
    print(f'\nTesting JPEG quality {quality}...')

    comp_ds = CompressedTestDataset(manifest_df, PROCESSED_DIR, jpeg_quality=quality)
    comp_loader = DataLoader(comp_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

    comp_results = full_evaluation(vit_model, freq_model, meta_model, comp_loader, device)

    comp_metrics = compute_metrics(comp_results['labels'], comp_results['fused'])
    vit_metrics = compute_metrics(comp_results['labels'], comp_results['vit'])
    freq_metrics = compute_metrics(comp_results['labels'], comp_results['freq'])

    compression_results[quality] = {
        'fused': comp_metrics,
        'vit': vit_metrics,
        'freq': freq_metrics,
    }

    print(f'  Fused AUC: {comp_metrics["AUC"]:.4f} | Acc: {comp_metrics["Accuracy"]:.4f}')

print('\nCompression robustness test complete.')

In [ ]:
# Plot AUC vs JPEG quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

qualities = sorted(compression_results.keys())

for name, color, marker in [('vit', '#2196F3', 'o'), ('freq', '#FF9800', 's'), ('fused', '#4CAF50', 'D')]:
    aucs = [compression_results[q][name]['AUC'] for q in qualities]
    accs = [compression_results[q][name]['Accuracy'] for q in qualities]
    display = {'vit': 'ViT', 'freq': 'Frequency', 'fused': 'Fused'}[name]

    axes[0].plot(qualities, aucs, f'-{marker}', color=color, linewidth=2, markersize=8, label=display)
    axes[1].plot(qualities, accs, f'-{marker}', color=color, linewidth=2, markersize=8, label=display)

axes[0].set_xlabel('JPEG Quality', fontsize=12)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('AUC vs JPEG Compression', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.5, 1.0)
axes[0].invert_xaxis()

axes[1].set_xlabel('JPEG Quality', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy vs JPEG Compression', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.5, 1.0)
axes[1].invert_xaxis()

plt.suptitle('Compression Robustness', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'compression_robustness.png'), dpi=150)
plt.show()

# Print table
print('\n--- Compression Robustness Table (Fused Model) ---')
print(f'{"Quality":>8s} {"AUC":>8s} {"Accuracy":>10s} {"F1":>8s}')
for q in qualities:
    m = compression_results[q]['fused']
    print(f'{q:>8d} {m["AUC"]:>8.4f} {m["Accuracy"]:>10.4f} {m["F1"]:>8.4f}')

---
## 5. GradCAM Explainability

Visualise which regions of the image the ViT branch focuses on when making predictions.  
This helps verify the model is using forensically meaningful features.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# FIX: use ClassifierOutputTarget which works reliably across grad-cam versions
# for single-logit binary models. BinaryClassifierOutputTarget expects a specific
# format that varies by library version.
try:
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_TARGET = ClassifierOutputTarget(0)  # single output neuron
except ImportError:
    from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
    GRADCAM_TARGET = BinaryClassifierOutputTarget(1)

# For ViT-L/14 at 224: 224/14 = 16 patches per side
target_layer = vit_model.backbone.blocks[-1].norm1

def reshape_transform(tensor, height=16, width=16):
    # FIX: assert the expected token count so misconfiguration fails loudly
    # rather than silently producing garbage heatmaps
    n_tokens = tensor.shape[1]
    expected = height * width + 1  # +1 for CLS
    assert n_tokens == expected, (
        f'Expected {expected} tokens (got {n_tokens}). '
        f'reshape_transform is hardcoded for 16x16 patches (ViT-L/14 @ 224).'
    )
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

cam = GradCAM(
    model=vit_model,
    target_layers=[target_layer],
    reshape_transform=reshape_transform
)

print('GradCAM initialised.')


In [ ]:
def generate_gradcam(image_path, model, cam_obj, device):
    """
    Generate GradCAM heatmap for a single image.
    Returns: (original image [0-1], heatmap overlay, P(fake)).
    """
    img = np.array(Image.open(image_path).convert('RGB'))
    img_float = img.astype(np.float32) / 255.0

    input_tensor = val_transform(image=img)['image'].unsqueeze(0).to(device)

    # Get prediction (no grad needed here)
    with torch.no_grad():
        logit = model(input_tensor).squeeze()
        prob = torch.sigmoid(logit).item()

    # Generate GradCAM (needs grad enabled internally)
    targets = [GRADCAM_TARGET]
    grayscale_cam = cam_obj(input_tensor=input_tensor, targets=targets)[0]

    overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)
    return img_float, overlay, prob

print('GradCAM generation function ready.')


In [ ]:
# Generate GradCAM for sample images from each category
# Select: 3 True Positives, 3 True Negatives, 2 False Positives, 2 False Negatives

test_df_reset = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)
fused_preds_binary = (results['fused'] >= 0.5).astype(int)

# Categorise predictions
tp_idx = np.where((labels == 1) & (fused_preds_binary == 1))[0]
tn_idx = np.where((labels == 0) & (fused_preds_binary == 0))[0]
fp_idx = np.where((labels == 0) & (fused_preds_binary == 1))[0]
fn_idx = np.where((labels == 1) & (fused_preds_binary == 0))[0]

print(f'True Positives:  {len(tp_idx)}')
print(f'True Negatives:  {len(tn_idx)}')
print(f'False Positives: {len(fp_idx)}')
print(f'False Negatives: {len(fn_idx)}')

In [ ]:
# Visualise GradCAM for selected samples

def plot_gradcam_grid(indices, title, test_df, data_root, model, cam_obj, device, max_show=4):
    """Plot GradCAM for up to max_show images."""
    n = min(len(indices), max_show)
    if n == 0:
        print(f'  No samples for: {title}')
        return

    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col, idx in enumerate(indices[:n]):
        row = test_df.iloc[idx]
        img_path = os.path.join(data_root, row['crop_path'])

        img_float, overlay, prob = generate_gradcam(img_path, model, cam_obj, device)

        # Original image
        axes[0, col].imshow(img_float)
        label_str = 'FAKE' if row['label'] == 1 else 'REAL'
        gen = row['generator'] if row['generator'] != 'none' else 'camera'
        axes[0, col].set_title(f'{label_str} ({gen})\nP(fake)={prob:.3f}', fontsize=10)
        axes[0, col].axis('off')

        # GradCAM overlay
        axes[1, col].imshow(overlay)
        axes[1, col].set_title('GradCAM', fontsize=10)
        axes[1, col].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'gradcam_{title.lower().replace(" ", "_")}.png'), dpi=150)
    plt.show()


plot_gradcam_grid(tp_idx, 'True Positives (Correctly Detected Fakes)', test_df_reset, PROCESSED_DIR, vit_model, cam, device)
plot_gradcam_grid(tn_idx, 'True Negatives (Correctly Identified Real)', test_df_reset, PROCESSED_DIR, vit_model, cam, device)
plot_gradcam_grid(fp_idx, 'False Positives (Real Misclassified as Fake)', test_df_reset, PROCESSED_DIR, vit_model, cam, device)
plot_gradcam_grid(fn_idx, 'False Negatives (Fake Misclassified as Real)', test_df_reset, PROCESSED_DIR, vit_model, cam, device)

---
## 6. Gradio Demo Interface

Interactive web demo: upload an image → get prediction + GradCAM heatmap.

In [ ]:
import gradio as gr

def predict_image(input_image):
    """
    Full inference pipeline for a single uploaded image.
    Returns: (result_text, gradcam_overlay)
    """
    if input_image is None:
        # Return shapes matching the two Gradio outputs
        return 'No image uploaded', None

    # Centre-crop to square, resize to 224
    img = Image.fromarray(input_image).convert('RGB')
    w, h = img.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    img = img.crop((left, top, left + crop_size, top + crop_size))
    img = img.resize((224, 224), Image.LANCZOS)
    img_array = np.array(img)
    img_float = img_array.astype(np.float32) / 255.0

    # RGB tensor
    rgb_tensor = val_transform(image=img_array)['image'].unsqueeze(0).to(device)

    # DCT tensor
    dct_array = compute_dct(img_array)
    dct_tensor = torch.tensor(dct_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    # Fused prediction
    with torch.no_grad():
        with autocast():
            vit_feat = vit_model.extract_features(rgb_tensor).float()
            freq_feat = freq_model.extract_features(dct_tensor).float()
            fused_feat = torch.cat([vit_feat, freq_feat], dim=1)
            prob = meta_model.predict_calibrated(fused_feat).squeeze().item()

    # GradCAM
    targets = [GRADCAM_TARGET]
    grayscale_cam = cam(input_tensor=rgb_tensor, targets=targets)[0]
    overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    label = 'AI-Generated (Fake)' if prob >= 0.5 else 'Real (Authentic)'
    confidence = prob if prob >= 0.5 else 1 - prob
    result_text = f'{label}\nConfidence: {confidence:.1%}\nP(fake): {prob:.4f}'

    return result_text, overlay

print('Prediction function ready.')


In [ ]:
# Launch Gradio demo

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(label='Upload Face Image', type='numpy'),
    outputs=[
        gr.Textbox(label='Prediction'),
        gr.Image(label='GradCAM Heatmap', type='numpy'),
    ],
    title='AI-Generated Face Detector',
    description=(
        'Upload a face image to determine if it is real or AI-generated. '
        'The system uses a dual-branch architecture: a CLIP ViT-L/14 for semantic analysis '
        'and an EfficientNet-B0 on DCT frequency representations, fused via a meta-classifier. '
        'The GradCAM heatmap shows which regions influenced the prediction.'
    ),
    examples=[
        # Add a few test images as examples if desired
    ],
    allow_flagging='never',
)

demo.launch(share=True, debug=True)

---
## 7. Save All Results

In [ ]:
# Save comprehensive results

final_results = {
    'test_metrics': {
        'vit': compute_metrics(labels, results['vit']),
        'freq': compute_metrics(labels, results['freq']),
        'fused': compute_metrics(labels, results['fused']),
    },
    'cross_generator': cross_gen_results,
    'compression_robustness': {
        str(q): {
            model: {k: v for k, v in metrics.items()}
            for model, metrics in qr.items()
        }
        for q, qr in compression_results.items()
    },
    'model_info': {
        'vit_epoch': vit_ckpt['epoch'],
        'vit_val_auc': vit_ckpt['val_auc'],
        'freq_epoch': freq_ckpt['epoch'],
        'freq_val_auc': freq_ckpt['val_auc'],
        'temperature': meta_model.temperature.item(),
    },
    'dataset_info': {
        'test_size': len(test_ds),
        'test_real': int((labels == 0).sum()),
        'test_fake': int((labels == 1).sum()),
    }
}

with open(os.path.join(RESULTS_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print('Results saved to:', RESULTS_DIR)
print('\nFiles in results directory:')
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Final summary

print('\n' + '=' * 60)
print('NOTEBOOK 3 COMPLETE — PROJECT SUMMARY')
print('=' * 60)

print(f'\n📊 Test Set Performance (Fused Model):')
fm = compute_metrics(labels, results['fused'])
print(f'   AUC:       {fm["AUC"]:.4f}')
print(f'   Accuracy:  {fm["Accuracy"]:.4f}')
print(f'   Precision: {fm["Precision"]:.4f}')
print(f'   Recall:    {fm["Recall"]:.4f}')
print(f'   F1:        {fm["F1"]:.4f}')

print(f'\n🔍 Cross-Generator (Flux — unseen):')
flux_mask = generators == 'flux'
if flux_mask.sum() > 0:
    flux_preds = results['fused'][flux_mask]
    flux_acc = accuracy_score(labels[flux_mask], (flux_preds >= 0.5).astype(int))
    print(f'   Accuracy on unseen Flux images: {flux_acc:.4f}')

print(f'\n📁 All outputs saved to Google Drive:')
print(f'   Checkpoints: {CHECKPOINT_DIR}')
print(f'   Results:     {RESULTS_DIR}')

print(f'\n✅ Project complete!')